In [1]:
#LA MODEL

import torch
import torch.nn as nn
import torchvision.models as models

class EmoNet(torch.nn.Module):
    def __init__(self, num_classes=8):
        super(EmoNet, self).__init__()

        self.conv1 = nn.Conv3d(in_channels=3, out_channels=16, kernel_size=(3, 3, 3), stride=1, padding=1)
        self.bn1 = nn.BatchNorm3d(16)  # Batch normalization for conv1
        self.conv2 = nn.Conv3d(in_channels=16, out_channels=32, kernel_size=(3, 3, 3), stride=1, padding=1)
        self.bn2 = nn.BatchNorm3d(32)  # Batch normalization for conv2
        self.conv3 = nn.Conv3d(in_channels=32, out_channels=64, kernel_size=(3, 3, 3), stride=1, padding=1)
        self.bn3 = nn.BatchNorm3d(64)  # Batch normalization for conv3
        self.conv4 = nn.Conv3d(in_channels=64, out_channels=128, kernel_size=(3, 3, 3), stride=1, padding=1)
        self.bn4 = nn.BatchNorm3d(128)  # Batch normalization for conv3

        # Max pooling layer
        self.pool = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2))
        self.dropout = nn.Dropout3d(p=0.3)

        # Define CNN blocks for audio processing
        self.conv2Dblock = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(p=0.3),
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=4, stride=4),
            nn.Dropout(p=0.3),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=4, stride=4),
            nn.Dropout(p=0.3),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=4, stride=4),
            nn.Dropout(p=0.3)
        )

        # LSTM block for audio
        self.lstm_maxpool = nn.MaxPool2d(kernel_size=[2,4], stride=[2,4])
        hidden_size = 128
        self.lstm = nn.LSTM(input_size=64, hidden_size=hidden_size, bidirectional=True, batch_first=True)
        self.dropout_lstm = nn.Dropout(0.1)
        self.attention_linear = nn.Linear(2 * hidden_size, 1)  # 2*hidden_size for bidirectional LSTM

        # Final linear layers
        self.out_linear = nn.Sequential(nn.Linear(228, 100),
                                        nn.ReLU(),
                                        nn.Dropout(0.3),
                                        nn.Linear(100, num_classes))

        self.video_fc1 = nn.Linear(128*3*12*4, 128)

        self.fc_lstm = nn.Sequential(nn.Linear(512, 100),
                                     nn.ReLU(),
                                     nn.Dropout(0.3))

    def forward(self, video, audio):
        # Process video
        video = self.pool(nn.ReLU()(self.bn1(self.conv1(video))))
        video = self.dropout(video)

        video = self.pool(nn.ReLU()(self.bn2(self.conv2(video))))
        video = self.dropout(video)

        video = self.pool(nn.ReLU()(self.bn3(self.conv3(video))))
        video = self.dropout(video)

        video = self.pool(nn.ReLU()(self.bn4(self.conv4(video))))
        video = self.dropout(video)

        video = video.view(32, -1)
        video_features = nn.ReLU()(self.video_fc1(video))
        #print(video_features.shape)

        # Process audio
        audio = audio.float()
        conv_embedding = self.conv2Dblock(audio)  # (b, channel, freq, time)
        conv_embedding = torch.flatten(conv_embedding, start_dim=1)  # Flattening except batch dimension
        #print("Conv embedding shape:", conv_embedding.shape)  # Should be [batch_size, conv_feature_dim]

        # LSTM processing
        x_reduced = self.lstm_maxpool(audio)  # Reduce dimensions
        x_reduced = torch.squeeze(x_reduced, 1)  # Remove dimension 1
        x_reduced = x_reduced.permute(0, 2, 1)  # Change shape to (b, t, freq)
        lstm_embedding, (h, c) = self.lstm(x_reduced)  # (b, t, hidden_size*2)
        lstm_embedding = self.dropout_lstm(lstm_embedding)
        batch_size, T, _ = lstm_embedding.shape

        # Compute attention weights
        attention_weights = [self.attention_linear(lstm_embedding[:, t, :]) for t in range(T)]
        attention_weights_norm = nn.functional.softmax(torch.stack(attention_weights, -1), -1)
        attention = torch.bmm(attention_weights_norm, lstm_embedding)  # (b, 1, 2*hidden_size)
        attention = torch.squeeze(attention, 1)  # Remove dimension 1

        # Concatenate embeddings
        audio_embedding = torch.cat([conv_embedding, attention], dim=1)
        audio_embedding = self.fc_lstm(audio_embedding)

        final_embedding = torch.cat([audio_embedding, video_features], dim=1)
        #print("Final embedding shape:", final_embedding.shape)  # Should be [batch_size, combined_feature_dim]

        # Pass through final linear layers
        return self.out_linear(final_embedding)


In [2]:
#La Training Loop
model = EmoNet()
num_epochs = 20
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
train_acc = []
test_acc = []
model.load_state_dict(torch.load('/content/drive/MyDrive/COSMOSPosterProject/EmoNet.pth'))
print(sum(p.numel() for p in model.parameters()))

for epoch in range(num_epochs):
    print(train_acc)
    print(test_acc)
    model.train()
    epoch_correct = 0
    epoch_total = 0
    running_loss = 0.0
    epoch_loss = []
    curr_time = time.time()
    for i, (video, audio, labels) in enumerate(train_dataloader):
        video, audio, labels = video.to(device), audio.to(device), labels.to(device)
        #print(audio.shape)
        audio = audio.unsqueeze(1)
        #print(audio.shape)

        # Zero the gradients
        optimizer.zero_grad()
        # Forward pass
        outputs = model(video, audio)
        #print(outputs.shape)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_loss.append(loss.item())

        # Calculate accuracy
        _, predicted = torch.max(outputs.data, 1)
        epoch_total += labels.size(0)
        epoch_correct += (predicted == labels).sum().item()

        # Print statistics
        running_loss += loss.item()
        if (i % 5 == 4):
            #print(f"Time: {time.time() - curr_time}")
            curr_time = time.time()
            print(f'Epoch {epoch + 1}, Training Batch {i + 1}, Loss: {running_loss / 5:.3f}, Accuracy: {epoch_correct / epoch_total * 100:.2f}%')
            running_loss = 0.0
            #print(epoch_loss)
            #print(epoch_acc)

    model.eval()
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for video, audio, labels in test_dataloader:
            video, audio, labels = video.to(device), audio.to(device), labels.to(device)
            audio = audio.unsqueeze(1)
            outputs = model(video, audio)
            _, predicted = torch.max(outputs.data, 1)
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()
    torch.save(model.state_dict(), '/content/drive/MyDrive/COSMOSPosterProject/EmoNet.pth')
    print(f'Epoch {epoch + 1}, Training Accuracy: {epoch_correct / epoch_total * 100:.2f}%')
    print(f'Epoch {epoch + 1}, Testing Accuracy: {test_correct / test_total * 100:.2f}%')

    train_acc.append(epoch_correct/epoch_total * 100)
    test_acc.append(test_correct/test_total * 100)

    # Save the model

    np.save('/content/drive/MyDrive/COSMOSPosterProject/EmoNet/data/densenet201_acc', train_acc)
    np.save('/content/drive/MyDrive/COSMOSPosterProject/EmoNet/data/densenet201_acc', test_acc)

print('Finished Training')